# DSA 210 – Analyzing and Predicting Procrastination Behavior Using Daily Digital Habits
**Milestone 1 – Data Collection, EDA & Hypothesis Testing**

This notebook covers:
1. Data loading and overview
2. Data cleaning and preprocessing
3. Exploratory Data Analysis (EDA)
4. Hypothesis Testing

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load dataset
df = pd.read_csv('procrastination_data.csv', parse_dates=['date'])
print(f'Dataset shape: {df.shape}')
df.head(10)

## 2. Data Cleaning & Preprocessing

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# Data types
print('\nData types:')
print(df.dtypes)

# Check for logical inconsistencies
print(f'\nRows where completed_tasks > planned_tasks: {(df.completed_tasks > df.planned_tasks).sum()}')

# Feature engineering
df['task_completion_rate'] = df['completed_tasks'] / df['planned_tasks']
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek >= 5

print('\nNew features added: task_completion_rate, day_of_week, is_weekend')
df[['date', 'planned_tasks', 'completed_tasks', 'task_completion_rate', 'day_of_week', 'is_weekend']].head()

In [ ]:
# Summary statistics
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Procrastination Score Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['date'], df['procrastination_score'], marker='o', linewidth=1.5, markersize=4, color='steelblue')
ax.axhline(df['procrastination_score'].mean(), linestyle='--', color='tomato', label=f'Mean = {df["procrastination_score"].mean():.2f}')
ax.set_title('Daily Procrastination Score Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Procrastination Score (1–10)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('procrastination_over_time.png', bbox_inches='tight')
plt.show()

### 3.2 Distribution of Key Variables

In [ ]:
features = ['sleep_hours', 'study_hours', 'social_media_time', 'screen_time', 'stress_level', 'procrastination_score']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flatten(), features):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Distribution of Key Variables', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('distributions.png', bbox_inches='tight')
plt.show()

### 3.3 Correlation Heatmap

In [ ]:
numeric_cols = ['sleep_hours', 'study_hours', 'screen_time', 'social_media_time',
                'planned_tasks', 'completed_tasks', 'stress_level',
                'task_completion_rate', 'procrastination_score']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 3.4 Scatter Plots: Key Predictors vs Procrastination Score

In [ ]:
predictors = ['social_media_time', 'sleep_hours', 'study_hours', 'stress_level']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, predictors):
    ax.scatter(df[col], df['procrastination_score'], alpha=0.6, color='steelblue', edgecolors='white', linewidths=0.4)
    m, b = np.polyfit(df[col], df['procrastination_score'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, m * x_line + b, color='tomato', linewidth=2)
    r, p = stats.pearsonr(df[col], df['procrastination_score'])
    ax.set_title(f'{col.replace("_", " ").title()}\nr={r:.2f}, p={p:.3f}', fontsize=10)
    ax.set_xlabel(col.replace('_', ' '))
    ax.set_ylabel('Procrastination Score')

plt.suptitle('Key Predictors vs Procrastination Score', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('scatter_plots.png', bbox_inches='tight')
plt.show()

### 3.5 Weekday vs Weekend Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, col in zip(axes, ['procrastination_score', 'social_media_time', 'study_hours']):
    sns.boxplot(data=df, x='is_weekend', y=col, ax=ax, palette=['steelblue', 'salmon'])
    ax.set_xticklabels(['Weekday', 'Weekend'])
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Weekday vs Weekend Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('weekday_weekend.png', bbox_inches='tight')
plt.show()

## 4. Hypothesis Testing

We test four hypotheses related to procrastination behavior.

### H1: High social media usage is associated with higher procrastination scores
- **H0**: There is no significant correlation between social media time and procrastination score.
- **H1**: Higher social media usage is positively correlated with procrastination score.
- **Test**: Pearson correlation (one-tailed)

In [ ]:
r, p_two = stats.pearsonr(df['social_media_time'], df['procrastination_score'])
p_one = p_two / 2  # one-tailed

print(f'Pearson r = {r:.3f}')
print(f'p-value (one-tailed) = {p_one:.4f}')
print(f'Result: {"Reject H0 – significant positive correlation" if p_one < 0.05 and r > 0 else "Fail to reject H0"}')

### H2: Less sleep is associated with higher procrastination
- **H0**: Sleep hours have no correlation with procrastination score.
- **H1**: Sleep hours are negatively correlated with procrastination score.
- **Test**: Pearson correlation (one-tailed)

In [ ]:
r, p_two = stats.pearsonr(df['sleep_hours'], df['procrastination_score'])
p_one = p_two / 2

print(f'Pearson r = {r:.3f}')
print(f'p-value (one-tailed) = {p_one:.4f}')
print(f'Result: {"Reject H0 – significant negative correlation" if p_one < 0.05 and r < 0 else "Fail to reject H0"}')

### H3: Procrastination score is higher on weekends than weekdays
- **H0**: Mean procrastination score is the same on weekdays and weekends.
- **H1**: Procrastination score is higher on weekends.
- **Test**: Independent samples t-test (one-tailed)

In [ ]:
weekday = df[df['is_weekend'] == False]['procrastination_score']
weekend = df[df['is_weekend'] == True]['procrastination_score']

t_stat, p_two = stats.ttest_ind(weekend, weekday)
p_one = p_two / 2

print(f'Weekday mean: {weekday.mean():.2f} | Weekend mean: {weekend.mean():.2f}')
print(f't-statistic = {t_stat:.3f}')
print(f'p-value (one-tailed) = {p_one:.4f}')
print(f'Result: {"Reject H0 – weekends have significantly higher procrastination" if p_one < 0.05 and t_stat > 0 else "Fail to reject H0"}')

### H4: Task completion rate is negatively correlated with procrastination score
- **H0**: Task completion rate has no relationship with procrastination score.
- **H1**: Higher task completion rate is associated with lower procrastination.
- **Test**: Spearman correlation (non-parametric, one-tailed)

In [ ]:
r_sp, p_two = stats.spearmanr(df['task_completion_rate'], df['procrastination_score'])
p_one = p_two / 2

print(f'Spearman r = {r_sp:.3f}')
print(f'p-value (one-tailed) = {p_one:.4f}')
print(f'Result: {"Reject H0 – significant negative correlation" if p_one < 0.05 and r_sp < 0 else "Fail to reject H0"}')

## 5. Summary of Findings

| Hypothesis | Test | Result |
|---|---|---|
| H1: Social media ↑ → Procrastination ↑ | Pearson r | To be confirmed by output |
| H2: Sleep ↓ → Procrastination ↑ | Pearson r | To be confirmed by output |
| H3: Weekends > Weekdays procrastination | t-test | To be confirmed by output |
| H4: Task completion ↑ → Procrastination ↓ | Spearman r | To be confirmed by output |

**Next step (Milestone 2):** Apply machine learning models (Linear Regression, Random Forest) to predict procrastination score from daily habits.